# Multi-channel ξ SPLM — α-initialisation sweep

This notebook sweeps **K-EMA decay initialisation** on the multi-channel-ξ SPLM
at the E9 scale-up configuration (d=256, L=8, K=4 channels, TinyStories).

## Cells

| CELL | α-init | Description |
|------|--------|-------------|
| `R6h0_baseline` | `[0.0, 0.5, 0.9, 0.99]` | R6.h.0 replication — hand-picked (paper baseline) |
| `R6h1_logspaced` | log-spaced, τ_max=100 | R6.h.1 replication — §4.2 Fix 2 |
| `uniform_narrow` | `[0.85, 0.90, 0.95, 0.99]` | All channels in the slow-decay band |
| `uniform_wide` | `[0.1, 0.4, 0.7, 0.95]` | Evenly spread across the [0,1] range |
| `fast_only` | `[0.0, 0.0, 0.1, 0.2]` | All channels fast-decaying (short memory) |
| `slow_only` | `[0.95, 0.97, 0.99, 0.999]` | All channels slow-decaying (long memory) |
| `learned_from_uniform` | `[0.25, 0.50, 0.75, 0.95]` | Learnable α from neutral init — let optimiser discover |
| `learned_from_zero` | `[0.0, 0.0, 0.0, 0.0]` | Learnable α from all-zero — coldest start |

All cells use `causal_force=True` (leak-free), `fixed_gamma=0.30`, pilot schedule
(4000 steps), and seed 0.  α values are **learnable** (except where noted) so the
final α values reveal where the optimiser converges from each initialisation.

## Hardware

- **A100 40GB / 80GB**: ~2.5 h per cell, ~20 h total for all 8 cells
- **L4 / T4**: ~4–5 h per cell (reduce `batch_size` if OOM)
- Select cells to run via the `CELLS_TO_RUN` list in the config cell below

## Results

Saved to Google Drive at `semsimula_alpha_sweep/results/<cell_name>/`

## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_alpha_sweep')
    REPO_PARENT = Path('/content')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_alpha_sweep'
    REPO_PARENT = Path.cwd().parent.parent.parent.parent

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)

In [ ]:
REPO_URL = 'https://github.com/dg-hub/semsimula-paper.git'
REPO_DIR = REPO_PARENT / 'semsimula-paper'

if IN_COLAB:
    if REPO_DIR.exists():
        print(f'Repo already cloned at {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                        str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'matplotlib', 'tiktoken', 'datasets'],
                   check=True)

SCRIPTS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
MULTIXI_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multixi'
CA_DIR      = REPO_DIR / 'notebooks' / 'conservative_arch'
assert SCRIPTS_DIR.exists(), f'Missing: {SCRIPTS_DIR}'
print('Scripts dir  :', SCRIPTS_DIR)

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    print('GPU: Apple MPS')
    DEVICE = 'mps'
else:
    print('WARNING: No GPU detected — training will be very slow')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Experiment configuration

Select which cells to run. Each cell trains the multi-ξ SPLM with a
different α initialisation. Set `CELLS_TO_RUN` to a subset if you want
to run only specific experiments.

In [ ]:
# ─── Shared hyperparameters (locked to E9/R6.h.0 config) ───
MODE          = 'pilot'      # 4000 steps
FIXED_GAMMA   = 0.30
XI_CHANNELS   = 4
XI_LEARNABLE  = True         # α values are learned during training
CAUSAL_FORCE  = True         # leak-free integrator
SEED          = 0
MAX_TRAIN_TOK = 5_000_000

# ─── Cell definitions ───
CELL_DEFS = {
    # ── Baseline replications ──
    'R6h0_baseline': {
        'alpha_init_mode': 'explicit',
        'alpha_inits': [0.0, 0.5, 0.9, 0.99],
        'desc': 'R6.h.0 replication — hand-picked α (paper baseline)',
    },
    'R6h1_logspaced': {
        'alpha_init_mode': 'log_spaced',
        'alpha_inits': None,   # computed from tau_max
        'tau_max': 100.0,
        'desc': 'R6.h.1 replication — log-spaced (§4.2 Fix 2)',
    },
    # ── Sweep: different α-init strategies ──
    'uniform_narrow': {
        'alpha_init_mode': 'explicit',
        'alpha_inits': [0.85, 0.90, 0.95, 0.99],
        'desc': 'All channels in the slow-decay band',
    },
    'uniform_wide': {
        'alpha_init_mode': 'explicit',
        'alpha_inits': [0.1, 0.4, 0.7, 0.95],
        'desc': 'Evenly spread across [0, 1]',
    },
    'fast_only': {
        'alpha_init_mode': 'explicit',
        'alpha_inits': [0.0, 0.0, 0.1, 0.2],
        'desc': 'All channels fast-decaying (short memory)',
    },
    'slow_only': {
        'alpha_init_mode': 'explicit',
        'alpha_inits': [0.95, 0.97, 0.99, 0.999],
        'desc': 'All channels slow-decaying (long memory)',
    },
    # ── Optimal α discovery (learnable from neutral starts) ──
    'learned_from_uniform': {
        'alpha_init_mode': 'explicit',
        'alpha_inits': [0.25, 0.50, 0.75, 0.95],
        'desc': 'Learnable α from neutral init — let optimiser discover',
    },
    'learned_from_zero': {
        'alpha_init_mode': 'explicit',
        'alpha_inits': [0.0, 0.0, 0.0, 0.0],
        'desc': 'Learnable α from all-zero — coldest start',
    },
}

# ─── Select cells to run (edit this list) ───
CELLS_TO_RUN = list(CELL_DEFS.keys())   # all cells; trim to run a subset

print(f'Will run {len(CELLS_TO_RUN)} cells:')
for name in CELLS_TO_RUN:
    d = CELL_DEFS[name]
    print(f'  {name:25s}  α={d["alpha_inits"]}  ({d["desc"]})')

## 4. Precompute logfreq surprisal (if needed)

In [ ]:
LOGFREQ_PATH = SCRIPTS_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'

if not LOGFREQ_PATH.exists():
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / 'compute_unigram_frequencies_tinystories.py')],
        cwd=str(SCRIPTS_DIR), check=True,
    )
    assert LOGFREQ_PATH.exists(), f'logfreq file not created at {LOGFREQ_PATH}'
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')

## 5. Run α-initialisation sweep

Each cell trains a fresh multi-ξ SPLM with the specified α-init.
Completed cells are **skipped** on re-run (idempotent).

In [ ]:
TRAINER = str(SCRIPTS_DIR / 'train_splm_em_ln_multixi_scaleup.py')

results = {}

for cell_name in CELLS_TO_RUN:
    cell_def = CELL_DEFS[cell_name]
    cell_results_dir = DRIVE_RESULTS / cell_name
    cell_results_dir.mkdir(parents=True, exist_ok=True)

    summary_glob = list(cell_results_dir.glob('*_summary.md'))
    if summary_glob:
        print(f'\n── {cell_name}: SKIP (already complete) ──')
        print(f'   {summary_glob[0]}')
        with open(summary_glob[0]) as f:
            for line in f:
                if 'Final val loss' in line or 'Final' in line:
                    print(f'   {line.strip()}')
        results[cell_name] = {'status': 'skipped'}
        continue

    print(f'\n{"═"*60}')
    print(f'  {cell_name}: {cell_def["desc"]}')
    print(f'  α-init: {cell_def["alpha_inits"]}')
    print(f'{"═"*60}')

    cmd = [
        sys.executable, TRAINER,
        '--mode', MODE,
        '--seed', str(SEED),
        '--fixed-gamma', str(FIXED_GAMMA),
        '--xi-channels', str(XI_CHANNELS),
        '--causal-force', 'true' if CAUSAL_FORCE else 'false',
        '--max-train-tokens', str(MAX_TRAIN_TOK),
        '--results-dir', str(cell_results_dir),
        '--tag-suffix', cell_name,
        '--xi-alpha-init-mode', cell_def['alpha_init_mode'],
        '--logfreq-path', str(LOGFREQ_PATH),
        '--device', DEVICE,
    ]

    if cell_def['alpha_init_mode'] == 'explicit' and cell_def['alpha_inits'] is not None:
        alpha_str = ','.join(str(a) for a in cell_def['alpha_inits'])
        cmd += ['--xi-alpha-inits', alpha_str]

    if cell_def['alpha_init_mode'] == 'log_spaced':
        cmd += ['--xi-tau-max', str(cell_def.get('tau_max', 100.0))]

    if not XI_LEARNABLE:
        cmd.append('--xi-frozen')

    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(SCRIPTS_DIR))
    elapsed = time.time() - t0

    if proc.returncode != 0:
        print(f'  ERROR: trainer exited with code {proc.returncode}')
        results[cell_name] = {'status': 'failed', 'returncode': proc.returncode}
        continue

    summary_files = list(cell_results_dir.glob('*_summary.md'))
    if summary_files:
        with open(summary_files[0]) as f:
            summary_text = f.read()
        print(f'\n{summary_text}')
    results[cell_name] = {
        'status': 'completed',
        'elapsed_min': elapsed / 60,
        'results_dir': str(cell_results_dir),
    }
    print(f'  Completed in {elapsed/60:.1f} min')

print(f'\n{"═"*60}')
print('Sweep complete.')
for name, r in results.items():
    print(f'  {name:25s}  {r["status"]}')

## 6. Results analysis — compare α trajectories and final PPL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

summary_data = []

for cell_name in CELL_DEFS:
    cell_dir = DRIVE_RESULTS / cell_name
    log_files = list(cell_dir.glob('*_training_log.jsonl'))
    if not log_files:
        continue

    rows = []
    with open(log_files[0]) as f:
        for line in f:
            rows.append(json.loads(line))
    if not rows:
        continue

    steps    = [r['step'] for r in rows]
    alphas_t = [r.get('xi_alphas', []) for r in rows]

    ckpt_files = list(cell_dir.glob('*_ckpt_latest.pt'))
    final_ppl = None
    final_alphas = None
    if ckpt_files:
        ckpt = torch.load(ckpt_files[0], map_location='cpu', weights_only=False)
        final_ppl = ckpt.get('final_val_ppl')
        final_alphas = ckpt.get('final_xi_alphas')

    init_alphas = CELL_DEFS[cell_name].get('alpha_inits')
    summary_data.append({
        'cell': cell_name,
        'init_alphas': init_alphas,
        'final_alphas': final_alphas,
        'final_ppl': final_ppl,
        'steps': steps,
        'alphas_trajectory': alphas_t,
    })

if not summary_data:
    print('No results found yet. Run the sweep cells first.')
else:
    print(f'Found results for {len(summary_data)} cells.\n')

    # ─── Summary table ───
    print(f'{"Cell":25s} {"Init α":30s} {"Final α":35s} {"Val PPL":>8s}')
    print('─' * 100)
    for d in sorted(summary_data, key=lambda x: x['final_ppl'] or 999):
        init_str = str(d['init_alphas']) if d['init_alphas'] else 'log-spaced'
        final_str = ', '.join(f'{a:.4f}' for a in d['final_alphas']) if d['final_alphas'] else '?'
        ppl_str = f'{d["final_ppl"]:.2f}' if d['final_ppl'] else '?'
        print(f'{d["cell"]:25s} {init_str:30s} [{final_str:33s}] {ppl_str:>8s}')

In [ ]:
# ─── Plot: α trajectories over training ───
if summary_data:
    n_cells = len(summary_data)
    fig, axes = plt.subplots(2, min(4, (n_cells + 1) // 2),
                             figsize=(16, 8), squeeze=False)
    axes_flat = axes.flatten()

    for idx, d in enumerate(summary_data):
        if idx >= len(axes_flat):
            break
        ax = axes_flat[idx]
        if d['alphas_trajectory'] and d['alphas_trajectory'][0]:
            K = len(d['alphas_trajectory'][0])
            for k in range(K):
                vals = [a[k] if k < len(a) else 0.0 for a in d['alphas_trajectory']]
                ax.plot(d['steps'], vals, label=f'α_{k}')
        ppl_str = f' (PPL {d["final_ppl"]:.2f})' if d['final_ppl'] else ''
        ax.set_title(f'{d["cell"]}{ppl_str}', fontsize=9)
        ax.set_xlabel('step')
        ax.set_ylabel('α')
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    for idx in range(len(summary_data), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle('α trajectories during training', fontsize=13)
    fig.tight_layout()
    fig.savefig(DRIVE_RESULTS / 'alpha_trajectories.png', dpi=150)
    plt.show()
    print(f'Saved: {DRIVE_RESULTS / "alpha_trajectories.png"}')

In [ ]:
# ─── Plot: final PPL bar chart ───
if summary_data:
    sorted_data = sorted(summary_data, key=lambda x: x['final_ppl'] or 999)
    names = [d['cell'] for d in sorted_data]
    ppls  = [d['final_ppl'] for d in sorted_data]

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#2ecc71' if p and p == min(pp for pp in ppls if pp) else '#3498db'
              for p in ppls]
    bars = ax.barh(names, ppls, color=colors, edgecolor='white')
    ax.set_xlabel('Val PPL')
    ax.set_title('Final Val PPL by α-initialisation (lower is better)')
    for bar, ppl in zip(bars, ppls):
        if ppl:
            ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                    f'{ppl:.2f}', va='center', fontsize=9)
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)
    fig.tight_layout()
    fig.savefig(DRIVE_RESULTS / 'ppl_comparison.png', dpi=150)
    plt.show()
    print(f'Saved: {DRIVE_RESULTS / "ppl_comparison.png"}')

## 7. Optimal α analysis

Compare where the learnable-α cells converge to — if different
initialisations converge to the same final α values, that's strong
evidence for a unique optimum.

In [ ]:
if summary_data:
    print('=== α convergence analysis ===')
    print()

    final_alphas_all = []
    for d in summary_data:
        if d['final_alphas']:
            final_alphas_all.append((d['cell'], d['final_alphas'], d['final_ppl']))

    if len(final_alphas_all) >= 2:
        best = min(final_alphas_all, key=lambda x: x[2] or 999)
        print(f'Best cell: {best[0]} (PPL {best[2]:.2f})')
        print(f'  Final α: [{" ,".join(f"{a:.4f}" for a in best[1])}]')
        print()

        sorted_by_alpha0 = sorted(final_alphas_all, key=lambda x: x[1][0])
        print(f'{"Cell":25s} {"α₀":>7s} {"α₁":>7s} {"α₂":>7s} {"α₃":>7s} {"PPL":>8s}')
        print('─' * 65)
        for name, alphas, ppl in sorted_by_alpha0:
            ppl_str = f'{ppl:.2f}' if ppl else '?'
            print(f'{name:25s} {alphas[0]:7.4f} {alphas[1]:7.4f} '
                  f'{alphas[2]:7.4f} {alphas[3]:7.4f} {ppl_str:>8s}')

        all_final = np.array([a[1] for a in final_alphas_all])
        mean_alpha = all_final.mean(axis=0)
        std_alpha  = all_final.std(axis=0)
        print()
        print(f'Mean final α: [{" ,".join(f"{m:.4f}" for m in mean_alpha)}]')
        print(f'Std  final α: [{" ,".join(f"{s:.4f}" for s in std_alpha)}]')
        print()
        if std_alpha.max() < 0.05:
            print('→ All cells converge to similar α values — strong evidence for unique optimum.')
        elif std_alpha.max() < 0.15:
            print('→ Moderate convergence — most cells agree on α direction.')
        else:
            print('→ High variance — α-init matters; the loss landscape may have multiple basins.')
    else:
        print('Need at least 2 completed cells for convergence analysis.')

## 8. Save consolidated results to Drive

In [ ]:
report = {
    'experiment': 'alpha_init_sweep',
    'config': {
        'mode': MODE,
        'fixed_gamma': FIXED_GAMMA,
        'xi_channels': XI_CHANNELS,
        'xi_learnable': XI_LEARNABLE,
        'causal_force': CAUSAL_FORCE,
        'seed': SEED,
        'max_train_tokens': MAX_TRAIN_TOK,
    },
    'cells': {},
}

for d in summary_data:
    report['cells'][d['cell']] = {
        'init_alphas': d['init_alphas'],
        'final_alphas': d['final_alphas'],
        'final_ppl': d['final_ppl'],
    }

report_path = DRIVE_RESULTS / 'alpha_sweep_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved: {report_path}')
print(f'\nAll results in: {DRIVE_RESULTS}')
if IN_COLAB:
    print('Results are persisted on Google Drive — safe to disconnect.')